# `run_pipeline.py` — Playground

Manual verification notebook for **the Pipeline Orchestrator** (Phase 4.6 — wiring only, no Claude of its own).

| Function | Status | Notes |
|---|---|---|
| `run_pipeline(candidate, portfolio_value, daily_pnl, shared=None)` | ✅ built | Chains Gates 1–5 for one candidate, stopping at the first failure |

**Output:** `{ticker, gates, final_decision}` — `final_decision` is one of `BUY`, `SKIP`, `BLOCKED_G1`, `BLOCKED_G2`, `BLOCKED_G3`, `BLOCKED_G4`, `FLAGGED_FOR_REVIEW`.

This is deliberately **thin** — unlike `backend/full_pipeline_playground.ipynb` (which runs Stage 1 + Stage 2 + a multi-candidate funnel with narrated markdown), this notebook only exercises `run_pipeline()`'s own contract: one function, one candidate at a time.

⚠️ **Live API + Claude calls.** Gates 2–4 each make one Claude (Haiku) call per candidate that reaches them. Needs `.env` keys: `ANTHROPIC_API_KEY` plus a news source (`ALPACA_API_KEY` + `ALPACA_SECRET_KEY`, or `NEWS_API_KEY`); `FINNHUB_API_KEY` for earnings/macro checks. Missing keys degrade gracefully — gates **block** rather than crash.

In [ ]:
import sys
import pathlib

# Anchor to backend/02_intelligence/ regardless of where the kernel started.
pipeline_dir = pathlib.Path('.').resolve()
if not (pipeline_dir / 'run_pipeline.py').exists():
    pipeline_dir = pathlib.Path('backend/02_intelligence/pipeline').resolve()

intelligence_dir = pipeline_dir.parent   # → backend/02_intelligence

for p in [str(intelligence_dir), str(pipeline_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from run_pipeline import run_pipeline
from gate1_hard_threat.hard_threat_gate1 import get_shared_market_data

---
## Happy path

One real candidate through the full chain. `final_decision` and the shape of `gates` vary with the live news window and market tape — eyeball, don't assert an exact verdict.

In [ ]:
candidate = {
    'ticker': 'AAPL',
    'sector': 'Electronic Technology',
    'price': 200.0,
    'atr': 5.0,
    'score': 3,
}

result = run_pipeline(candidate, portfolio_value=100_000.0, daily_pnl=0.0)
print('final_decision:', result['final_decision'])
print('gates ran:', list(result['gates'].keys()))
result

---
## Parameter variations

A couple more real tickers/sectors, plus the batch-reuse pattern: fetch `shared` market data **once** and pass it into multiple `run_pipeline()` calls, exactly as a future nightly-loop caller (Phase 8 `bot.py`) would, instead of refetching VIX/SPY/macro per candidate.

In [ ]:
shared = get_shared_market_data()   # fetched once, reused below
print('shared market data:', shared)

for cand in [
    {'ticker': 'JPM', 'sector': 'Finance', 'price': 250.0, 'atr': 4.0, 'score': 2},
    {'ticker': 'XOM', 'sector': 'Energy Minerals', 'price': 110.0, 'atr': 2.5, 'score': 3},
]:
    r = run_pipeline(cand, portfolio_value=100_000.0, daily_pnl=0.0, shared=shared)
    print(f"{cand['ticker']:<5} final_decision={r['final_decision']:<20} gates={list(r['gates'].keys())}")

---
## Failure path — forced daily loss-limit block

Deterministic, no network dependence: `daily_pnl=-5,000` on a `100,000` portfolio breaches the 3% daily loss limit, so Gate 1 blocks before any Claude call or news fetch. `gates` should contain only `gate1`.

In [ ]:
blocked = run_pipeline(candidate, portfolio_value=100_000.0, daily_pnl=-5_000.0)
print('final_decision:', blocked['final_decision'])
print('block_reason:', blocked['gates']['gate1']['block_reason'])
assert blocked['final_decision'] == 'BLOCKED_G1'
assert blocked['gates']['gate1']['block_reason'] == 'loss_limit'
assert 'gate2' not in blocked['gates']
print('[pipeline] forced loss_limit block correctly triggered ✅')

---
## Phase 4.7 — real scanner candidates, not hand-typed

Closes the actual gap 4.7 was tracking: everything above uses hand-typed candidate dicts.
This sources candidates from the real `run_scan()` output (building the watchlist live via
`run_universe_filter()` if one doesn't exist yet) and live portfolio state from
`alpaca_executor.py`, then pushes each through `run_pipeline()` — the same function calls
above, just with real inputs instead of synthetic ones.


In [ ]:
import pathlib as _pathlib

scanner_dir = _pathlib.Path('backend/01_scanner').resolve()
if not scanner_dir.exists():
    scanner_dir = (intelligence_dir.parent / '01_scanner').resolve()
execution_dir = (scanner_dir.parent / '04_execution').resolve()

for p in [str(scanner_dir), str(execution_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from universe_filter import run_universe_filter, WATCHLIST_PATH
from momentum_scanner import run_scan
from alpaca_executor import get_portfolio_value, get_daily_pnl

watchlist_path = scanner_dir / WATCHLIST_PATH
if not watchlist_path.exists():
    print(f'no watchlist at {watchlist_path} — building one live via run_universe_filter()')
    import os
    cwd = os.getcwd()
    os.chdir(scanner_dir)
    try:
        run_universe_filter()
    finally:
        os.chdir(cwd)

candidates = run_scan(top_n=3)
print(candidates)


In [ ]:
portfolio_value = get_portfolio_value() or 100_000.0
daily_pnl = get_daily_pnl() or 0.0
shared_real = get_shared_market_data()

for _, row in candidates.iterrows():
    real_candidate = {
        'ticker': row['ticker'],
        'sector': row['sector'],
        'price': float(row['price']),
        'atr': float(row['atr']),
        'score': int(row['score']),
    }
    result = run_pipeline(real_candidate, portfolio_value=portfolio_value, daily_pnl=daily_pnl, shared=shared_real)
    print(f"{result['ticker']}: {result['final_decision']}")


---
## Free-play

Try your own candidate, portfolio state, or edge cases below.